# Chứng minh: Cùng mô hình, cùng seed - Kết quả khác nhau trên GPU khác nhau

**Mục tiêu**: Chứng minh rằng ngay cả khi train cùng mô hình, cùng seed, kết quả vẫn khác nhau khi chạy trên các GPU khác nhau (T4, L4, A100, RTX 3060).

**Vấn đề**: Loss và accuracy khác nhau giữa các GPU, mặc dù:
- Cùng mô hình architecture
- Cùng random seed
- Cùng dataset và preprocessing
- Cùng hyperparameters


## 1. Thiết lập môi trường và import thư viện


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import json
from datetime import datetime
import os

# Kiểm tra GPU
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"cuDNN version: {torch.backends.cudnn.version()}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Compute Capability: {torch.cuda.get_device_capability(0)}")
    
    # Kiểm tra TF32 support
    print(f"\nTF32 enabled (cuDNN): {torch.backends.cudnn.allow_tf32}")
    # Kiểm tra cuBLAS TF32 (có thể không có trong một số phiên bản PyTorch)
    if hasattr(torch.backends, 'cublas'):
        print(f"TF32 enabled (cuBLAS): {torch.backends.cublas.allow_tf32}")
    else:
        print("TF32 enabled (cuBLAS): N/A (không hỗ trợ trong phiên bản PyTorch này)")


## 2. Hàm thiết lập seed và môi trường


In [ ]:
def set_seed(seed=42):
    """Thiết lập seed cho tất cả các RNG để đảm bảo reproducibility"""
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Các cài đặt deterministic (sẽ test riêng trong notebook khác)
    # torch.backends.cudnn.deterministic = True
    # torch.backends.cudnn.benchmark = False
    # torch.use_deterministic_algorithms(True)
    
def setup_environment(tf32_enabled=True, amp_enabled=False, num_workers=4):
    """
    Thiết lập môi trường với các cấu hình khác nhau
    
    Args:
        tf32_enabled: Bật/tắt TF32 (mặc định bật trên Ampere+)
        amp_enabled: Bật/tắt Automatic Mixed Precision
        num_workers: Số worker cho DataLoader
    """
    torch.backends.cudnn.allow_tf32 = tf32_enabled
    # cuBLAS TF32 (có thể không có trong một số phiên bản PyTorch)
    if hasattr(torch.backends, 'cublas'):
        torch.backends.cublas.allow_tf32 = tf32_enabled
    
    config = {
        'tf32_enabled': tf32_enabled,
        'amp_enabled': amp_enabled,
        'num_workers': num_workers,
        'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
        'gpu_capability': torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None
    }
    
    return config


## 3. Định nghĩa mô hình đơn giản (CNN cho CIFAR-10)


In [ ]:
class SimpleCNN(nn.Module):
    """Mô hình CNN đơn giản với BatchNorm và Dropout để dễ thấy sự khác biệt"""
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.5)
        
        # CIFAR-10: 32x32 -> 16x16 -> 8x8 -> 4x4
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)
    
    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = x.view(-1, 128 * 4 * 4)
        x = self.dropout(x)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


## 4. Load dữ liệu CIFAR-10


In [ ]:
def get_cifar10_loaders(batch_size=128, num_workers=4, seed=42):
    """Load CIFAR-10 với preprocessing chuẩn"""
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
    ])
    
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
    ])
    
    # Set seed cho DataLoader
    def worker_init_fn(worker_id):
        np.random.seed(seed + worker_id)
    
    trainset = torchvision.datasets.CIFAR10(
        root='./data', train=True, download=True, transform=transform_train
    )
    trainloader = DataLoader(
        trainset, batch_size=batch_size, shuffle=True, 
        num_workers=num_workers, worker_init_fn=worker_init_fn if num_workers > 0 else None
    )
    
    testset = torchvision.datasets.CIFAR10(
        root='./data', train=False, download=True, transform=transform_test
    )
    testloader = DataLoader(
        testset, batch_size=batch_size, shuffle=False, num_workers=num_workers
    )
    
    return trainloader, testloader


## 5. Hàm training với logging chi tiết


In [ ]:
def train_one_epoch(model, trainloader, criterion, optimizer, device, amp_enabled=False):
    """Train một epoch và trả về loss trung bình"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    scaler = torch.cuda.amp.GradScaler() if amp_enabled else None
    
    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        if amp_enabled and scaler is not None:
            with torch.cuda.amp.autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    epoch_loss = running_loss / len(trainloader)
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

def evaluate(model, testloader, criterion, device):
    """Evaluate trên test set"""
    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    test_loss = test_loss / len(testloader)
    test_acc = 100. * correct / total
    return test_loss, test_acc


## 6. Hàm train đầy đủ với logging


In [ ]:
def train_model(config, seed=42, num_epochs=10, save_results=True):
    """
    Train mô hình với cấu hình cụ thể và lưu kết quả
    
    Returns:
        dict: Kết quả training bao gồm loss, accuracy theo epoch
    """
    # Setup
    set_seed(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Load data
    trainloader, testloader = get_cifar10_loaders(
        batch_size=128, 
        num_workers=config['num_workers'],
        seed=seed
    )
    
    # Model
    model = SimpleCNN(num_classes=10).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    # Training history
    history = {
        'train_loss': [],
        'train_acc': [],
        'test_loss': [],
        'test_acc': [],
        'config': config,
        'seed': seed
    }
    
    print(f"\n{'='*60}")
    print(f"Training với cấu hình:")
    print(f"  GPU: {config['gpu_name']}")
    print(f"  TF32: {config['tf32_enabled']}")
    print(f"  AMP: {config['amp_enabled']}")
    print(f"  Num Workers: {config['num_workers']}")
    print(f"  Seed: {seed}")
    print(f"{'='*60}\n")
    
    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(
            model, trainloader, criterion, optimizer, device, 
            amp_enabled=config['amp_enabled']
        )
        test_loss, test_acc = evaluate(model, testloader, criterion, device)
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)
        
        print(f"Epoch {epoch+1}/{num_epochs}:")
        print(f"  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"  Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%")
    
    # Lưu kết quả
    if save_results:
        os.makedirs('results', exist_ok=True)
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        gpu_name_safe = config['gpu_name'].replace(' ', '_').replace('/', '_')
        filename = f"results/train_{gpu_name_safe}_tf32_{config['tf32_enabled']}_amp_{config['amp_enabled']}_{timestamp}.json"
        
        with open(filename, 'w') as f:
            json.dump(history, f, indent=2)
        print(f"\nKết quả đã lưu: {filename}")
    
    return history


## 7. Chạy thử nghiệm: Cùng seed, khác GPU (mô phỏng)

### 7.1 Test với cấu hình mặc định (TF32 bật, nếu GPU hỗ trợ)

**Lưu ý**: Để thấy sự khác biệt rõ ràng, bạn cần chạy notebook này trên các GPU khác nhau:
- T4 (không hỗ trợ TF32)
- L4 (không hỗ trợ TF32)
- A100 (hỗ trợ TF32)
- RTX 3060 (hỗ trợ TF32)

Hoặc có thể test trên cùng GPU nhưng với các cấu hình khác nhau (TF32 on/off).


In [ ]:
# Test 1: Cấu hình mặc định (TF32 bật nếu GPU hỗ trợ)
config_default = setup_environment(tf32_enabled=True, amp_enabled=False, num_workers=4)
history_default = train_model(config_default, seed=42, num_epochs=10)


### 7.2 Test với TF32 tắt (để so sánh)


In [ ]:
# Test 2: Tắt TF32
config_no_tf32 = setup_environment(tf32_enabled=False, amp_enabled=False, num_workers=4)
history_no_tf32 = train_model(config_no_tf32, seed=42, num_epochs=10)


### 7.3 Test với num_workers=0 (để loại bỏ non-determinism từ DataLoader)


In [ ]:
# Test 3: num_workers=0
config_no_workers = setup_environment(tf32_enabled=True, amp_enabled=False, num_workers=0)
history_no_workers = train_model(config_no_workers, seed=42, num_epochs=10)


## 8. So sánh kết quả


In [ ]:
def compare_results(histories, labels):
    """So sánh và visualize kết quả từ nhiều lần train"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Train Loss
    ax = axes[0, 0]
    for hist, label in zip(histories, labels):
        ax.plot(hist['train_loss'], label=label, marker='o')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Train Loss')
    ax.set_title('Train Loss Comparison')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Test Loss
    ax = axes[0, 1]
    for hist, label in zip(histories, labels):
        ax.plot(hist['test_loss'], label=label, marker='s')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Test Loss')
    ax.set_title('Test Loss Comparison')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Train Accuracy
    ax = axes[1, 0]
    for hist, label in zip(histories, labels):
        ax.plot(hist['train_acc'], label=label, marker='o')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Train Accuracy (%)')
    ax.set_title('Train Accuracy Comparison')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Test Accuracy
    ax = axes[1, 1]
    for hist, label in zip(histories, labels):
        ax.plot(hist['test_acc'], label=label, marker='s')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Test Accuracy (%)')
    ax.set_title('Test Accuracy Comparison')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('results/comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # In bảng so sánh
    print("\n" + "="*80)
    print("BẢNG SO SÁNH KẾT QUẢ CUỐI CÙNG")
    print("="*80)
    print(f"{'Cấu hình':<40} {'Train Loss':<12} {'Test Loss':<12} {'Test Acc':<12}")
    print("-"*80)
    for hist, label in zip(histories, labels):
        final_train_loss = hist['train_loss'][-1]
        final_test_loss = hist['test_loss'][-1]
        final_test_acc = hist['test_acc'][-1]
        print(f"{label:<40} {final_train_loss:<12.4f} {final_test_loss:<12.4f} {final_test_acc:<12.2f}%")
    print("="*80)


In [ ]:
# So sánh các kết quả
histories_to_compare = [history_default, history_no_tf32, history_no_workers]
labels_to_compare = [
    f"Default (TF32={config_default['tf32_enabled']})",
    f"No TF32",
    f"No Workers"
]

compare_results(histories_to_compare, labels_to_compare)


## 9. Kết luận

**Quan sát**:
- Ngay cả với cùng seed, cùng mô hình, kết quả vẫn có thể khác nhau
- Sự khác biệt có thể đến từ:
  - TF32 (trên GPU Ampere+)
  - cuDNN kernel selection
  - DataLoader với nhiều workers
  - Thứ tự tính toán song song không cố định

**Điều quan trọng**:
- So sánh **xu hướng** (trend), không phải giá trị tuyệt đối
- Nếu loss có cùng xu hướng giảm và validation performance ở cùng cấp độ → **Statistical reproducibility** (chấp nhận được)
- Nếu quỹ đạo học khác hẳn → cần kiểm tra cấu hình

**Ghi chú**: Để test trên GPU khác, chạy lại notebook này trên hệ thống khác và so sánh file JSON trong thư mục `results/`.
